# Legacy Gold Annotation, Diagnostic Inspection

Purpose: produce concrete statistics about the legacy `.xlsx` files so
their actual state can be discussed with whoever owns them, before any
attempt to use them as gold annotations.

## What this notebook does

1. Loads every `.xlsx` in the legacy directory.
2. For each sheet of each file, reports:
   - Row count and column structure
   - Which columns are "standard" entry-data columns vs. extras
   - How many cells in extra columns are populated (candidate label cells)
   - How many rows have suspicious entry-data patterns (placeholders,
     all-empty rows, etc.)
3. Inspects cell formatting (fill colors, fonts) to detect annotations
   that may be encoded visually rather than as values.
4. Produces a single CSV summary you can share for discussion.

## What this notebook does NOT do

- It does not produce normalized gold annotations
- It does not assume any particular meaning for the data
- It does not reconcile the legacy schema to the new annotation schema

These steps come AFTER you've confirmed (via discussion with the file
owner) what the data actually represents.

## Output

`../csvAnalysis/legacy_gold_diagnostic/_summary.csv`: one row per
(file, sheet) with all the diagnostic metrics.

## 1. Configuration

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
import warnings
warnings.filterwarnings('ignore')

LEGACY_DIR = Path("../csvAnalysis/gold_entries/legacy") 
OUT_DIR    = Path("../csvAnalysis/legacy_gold_diagnostic")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Standard entry-data columns the legacy files share
STANDARD_COLS = [
    "kata_asal", "kata_dasar", "kata_tujuan", "makna",
    "form", "kalimat_asal", "kalimat_tujuan",
]

# Patterns that suggest a row is a placeholder/empty rather than real data
PLACEHOLDER_VALUES = {"-", "--", "", "nan", "NaN", "N/A", "n/a", "?"}

print(f"Legacy dir: {LEGACY_DIR.resolve()}")
print(f"Output dir: {OUT_DIR.resolve()}")
print(f"Standard columns: {STANDARD_COLS}")

assert LEGACY_DIR.exists(), (
    f"Legacy directory missing. Place the .xlsx files in {LEGACY_DIR} or "
    f"adjust the LEGACY_DIR path above."
)

xlsx_files = sorted(LEGACY_DIR.glob("*.xlsx"))
print(f"\nFound {len(xlsx_files)} .xlsx files:")
for f in xlsx_files:
    print(f"  {f.name}")

Legacy dir: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\csvAnalysis\gold_entries\legacy
Output dir: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\csvAnalysis\legacy_gold_diagnostic
Standard columns: ['kata_asal', 'kata_dasar', 'kata_tujuan', 'makna', 'form', 'kalimat_asal', 'kalimat_tujuan']

Found 6 .xlsx files:
  Gold Entries Tahap 1_Hafiz.xlsx
  Gold Entries Tahap 1_Niken.xlsx
  Gold Entries Tahap 1_Yusuf.xlsx
  GoldEntries_Tahap2_Hafiz.xlsx
  GoldEntries_Tahap2_Niken.xlsx
  GoldEntries_Tahap2_Qonita.xlsx


## 2. Per-sheet diagnostics

For each (file, sheet) pair, compute:
- **Row count**: total number of rows
- **Standard columns present**: out of the 7 expected, how many appear
- **Extra columns**: columns beyond the 7 standard ones (these are the
  candidate label columns)
- **Header-like extras**: extras whose name is a number (these may be
  annotator-tracking columns, not labels)
- **Cells filled in extras**: how many cells in extras have non-null
  values (= candidate annotations)
- **Rows with all-extras-empty**: rows where no extra column has a value
- **Rows with placeholder kata_asal**: rows where the headword is `-`,
  `--`, or empty
- **Rows with content in kalimat_asal**: rows where parcor-side text is
  populated (may help if some sheets actually contain parcor-style data)

In [4]:
def is_placeholder(value) -> bool:
    if pd.isna(value): return True
    if isinstance(value, str) and value.strip() in PLACEHOLDER_VALUES:
        return True
    return False


def header_looks_numeric(name) -> bool:
    """True if column header is a number (e.g., 4, 127, 0)."""
    try:
        int(str(name))
        return True
    except (ValueError, TypeError):
        return False


def diagnose_sheet(path: Path, sheet_name: str) -> dict:
    df = pd.read_excel(path, sheet_name=sheet_name)
    n_rows = len(df)
    n_cols = len(df.columns)

    standard_present = [c for c in STANDARD_COLS if c in df.columns]
    extras = [c for c in df.columns if c not in STANDARD_COLS]

    numeric_extras   = [c for c in extras if header_looks_numeric(c)]
    named_extras     = [c for c in extras if not header_looks_numeric(c)]

    # Count cells filled in any extra column
    if extras:
        extras_filled_per_row = df[extras].notna().any(axis=1)
        n_rows_with_extras_filled = int(extras_filled_per_row.sum())
        n_cells_filled_in_extras = int(df[extras].notna().sum().sum())
    else:
        n_rows_with_extras_filled = 0
        n_cells_filled_in_extras  = 0

    # Count rows where named extras have content (these are most likely actual labels)
    if named_extras:
        named_filled_per_row = df[named_extras].notna().any(axis=1)
        n_rows_with_named_extras_filled = int(named_filled_per_row.sum())
    else:
        n_rows_with_named_extras_filled = 0

    # Check standard column patterns
    if "kata_asal" in df.columns:
        n_kata_asal_placeholder = int(df["kata_asal"].apply(is_placeholder).sum())
    else:
        n_kata_asal_placeholder = -1

    if "kalimat_asal" in df.columns:
        n_kalimat_asal_filled = int(df["kalimat_asal"].apply(
            lambda v: not is_placeholder(v)
        ).sum())
    else:
        n_kalimat_asal_filled = -1

    return {
        "file":                              path.name,
        "sheet":                             sheet_name,
        "n_rows":                            n_rows,
        "n_cols":                            n_cols,
        "standard_cols_present":             len(standard_present),
        "n_extras_total":                    len(extras),
        "n_extras_numeric":                  len(numeric_extras),
        "n_extras_named":                    len(named_extras),
        "named_extras_list":                 ",".join(str(c) for c in named_extras),
        "numeric_extras_list":               ",".join(str(c) for c in numeric_extras),
        "n_rows_with_any_extras_filled":     n_rows_with_extras_filled,
        "n_rows_with_named_extras_filled":   n_rows_with_named_extras_filled,
        "n_cells_filled_in_extras":          n_cells_filled_in_extras,
        "n_kata_asal_placeholder":           n_kata_asal_placeholder,
        "n_kalimat_asal_filled":             n_kalimat_asal_filled,
    }


sheet_summaries = []
for path in xlsx_files:
    wb = load_workbook(path, read_only=True, data_only=True)
    for sheet_name in wb.sheetnames:
        try:
            summary = diagnose_sheet(path, sheet_name)
            sheet_summaries.append(summary)
        except Exception as e:
            sheet_summaries.append({
                "file": path.name,
                "sheet": sheet_name,
                "error": str(e),
            })

sheet_df = pd.DataFrame(sheet_summaries)
print(f"Diagnosed {len(sheet_df)} sheets across {len(xlsx_files)} files\n")
print(sheet_df[[
    "file", "sheet", "n_rows", "n_extras_named",
    "n_rows_with_named_extras_filled", "n_kalimat_asal_filled",
    "n_kata_asal_placeholder",
]].to_string(index=False))

Diagnosed 30 sheets across 6 files

                           file sheet  n_rows  n_extras_named  n_rows_with_named_extras_filled  n_kalimat_asal_filled  n_kata_asal_placeholder
Gold Entries Tahap 1_Hafiz.xlsx    34     164               0                                0                      0                      160
Gold Entries Tahap 1_Hafiz.xlsx    42     164               0                                0                      1                      152
Gold Entries Tahap 1_Hafiz.xlsx    54     164               0                                0                      9                      150
Gold Entries Tahap 1_Hafiz.xlsx    71     164               0                                0                     12                      147
Gold Entries Tahap 1_Hafiz.xlsx    89     164               0                                0                     23                      139
Gold Entries Tahap 1_Niken.xlsx    34     164               0                                0            

## 3. Cell-formatting inspection

Some annotation conventions encode labels as cell formatting rather than
cell values: highlighted rows, colored cells, bold text, strikethrough.
Check whether any of these conventions are in use in the legacy files.

This iterates each sheet and counts cells with non-default fill colors,
font colors, or strikethrough. If meaningful counts appear (e.g., dozens
or hundreds of cells per sheet), the labels may be encoded visually.

In [5]:
def inspect_formatting(path: Path, sheet_name: str, max_rows: int = 200) -> dict:
    """
    Count cells with non-default formatting.
    Limited to first max_rows for speed (full files run into thousands
    of cells).
    """
    wb = load_workbook(path, data_only=True)
    ws = wb[sheet_name]

    n_filled_cells = 0
    n_colored_font = 0
    n_strikethrough = 0
    n_bold = 0
    fill_examples = []
    font_examples = []

    for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=max_rows + 1), start=2):
        for cell in row:
            # Fill color (background)
            if cell.fill and cell.fill.fgColor:
                rgb = cell.fill.fgColor.rgb
                if rgb and rgb not in ("00000000", None) and rgb != "FFFFFFFF":
                    n_filled_cells += 1
                    if len(fill_examples) < 3:
                        fill_examples.append((row_idx, cell.column_letter, str(cell.value)[:40], rgb))

            # Font color
            if cell.font and cell.font.color:
                color = cell.font.color.rgb
                if color and color not in ("00000000", None, "FF000000"):
                    n_colored_font += 1
                    if len(font_examples) < 3:
                        font_examples.append((row_idx, cell.column_letter, str(cell.value)[:40], color))

            # Strikethrough
            if cell.font and cell.font.strike:
                n_strikethrough += 1

            # Bold
            if cell.font and cell.font.bold:
                n_bold += 1

    return {
        "file":              path.name,
        "sheet":             sheet_name,
        "checked_rows":      max_rows,
        "n_cells_filled":    n_filled_cells,
        "n_colored_font":    n_colored_font,
        "n_strikethrough":   n_strikethrough,
        "n_bold":            n_bold,
        "fill_examples":     str(fill_examples) if fill_examples else "",
        "font_examples":     str(font_examples) if font_examples else "",
    }


print("Inspecting cell formatting (first 200 rows of each sheet)...\n")
formatting_summaries = []
for path in xlsx_files:
    wb = load_workbook(path, read_only=True, data_only=True)
    for sheet_name in wb.sheetnames:
        try:
            fmt = inspect_formatting(path, sheet_name)
            formatting_summaries.append(fmt)
        except Exception as e:
            formatting_summaries.append({
                "file": path.name,
                "sheet": sheet_name,
                "error": str(e),
            })

format_df = pd.DataFrame(formatting_summaries)
print(format_df[[
    "file", "sheet", "n_cells_filled", "n_colored_font",
    "n_strikethrough", "n_bold",
]].to_string(index=False))

# If any sheet has substantial formatting, print examples
print("\n=== Sheets with notable formatting (>10 instances) ===")
notable = format_df[
    (format_df["n_cells_filled"] > 10) |
    (format_df["n_colored_font"] > 10) |
    (format_df["n_strikethrough"] > 10)
]
if len(notable) == 0:
    print("(none — formatting is unlikely to be carrying annotation information)")
else:
    for _, row in notable.iterrows():
        print(f"  {row['file']} / sheet {row['sheet']}: filled={row['n_cells_filled']}, "
              f"colored_font={row['n_colored_font']}, strikethrough={row['n_strikethrough']}")
        if row.get("fill_examples"):
            print(f"    fill examples: {row['fill_examples']}")

Inspecting cell formatting (first 200 rows of each sheet)...

                           file sheet  n_cells_filled  n_colored_font  n_strikethrough  n_bold
Gold Entries Tahap 1_Hafiz.xlsx    34               0            1038                0       0
Gold Entries Tahap 1_Hafiz.xlsx    42               0            1038                0       0
Gold Entries Tahap 1_Hafiz.xlsx    54               0            1034                0       0
Gold Entries Tahap 1_Hafiz.xlsx    71               0            1038                0       0
Gold Entries Tahap 1_Hafiz.xlsx    89               0            1038                0       0
Gold Entries Tahap 1_Niken.xlsx    34               0            1038                0       0
Gold Entries Tahap 1_Niken.xlsx    42               0            1038                0       0
Gold Entries Tahap 1_Niken.xlsx    54               0            1038                0       0
Gold Entries Tahap 1_Niken.xlsx    71               0            1036              

## 4. File-level rollup and questions to take to file owner

Aggregates the per-sheet stats to one row per file, then prints concrete
questions the file owner needs to answer.

In [6]:
file_rollup = sheet_df.groupby("file").agg({
    "n_rows":                            "sum",
    "n_rows_with_any_extras_filled":     "sum",
    "n_rows_with_named_extras_filled":   "sum",
    "n_cells_filled_in_extras":          "sum",
    "n_kata_asal_placeholder":           "sum",
    "n_kalimat_asal_filled":             "sum",
}).reset_index()

file_rollup["pct_rows_named_extras_filled"] = (
    file_rollup["n_rows_with_named_extras_filled"] / file_rollup["n_rows"] * 100
).round(1)

print("=== File-level rollup ===\n")
print(file_rollup.to_string(index=False))

# Save the combined diagnostic
combined = sheet_df.merge(format_df, on=["file", "sheet"], suffixes=("", "_fmt"))
combined.to_csv(OUT_DIR / "_summary.csv", index=False)
print(f"\nWrote: {OUT_DIR / '_summary.csv'}")

=== File-level rollup ===

                           file  n_rows  n_rows_with_any_extras_filled  n_rows_with_named_extras_filled  n_cells_filled_in_extras  n_kata_asal_placeholder  n_kalimat_asal_filled  pct_rows_named_extras_filled
Gold Entries Tahap 1_Hafiz.xlsx     820                              0                                0                         0                      748                     45                           0.0
Gold Entries Tahap 1_Niken.xlsx     820                              0                                0                         0                      773                     19                           0.0
Gold Entries Tahap 1_Yusuf.xlsx     820                              0                                0                         0                      767                     28                           0.0
  GoldEntries_Tahap2_Hafiz.xlsx     831                              0                                0                         0            

## 5. Questions to ask the file owner

Based on the diagnostics above, these are the questions worth raising:

### If `n_rows_with_named_extras_filled` is near zero across all files

**The labels appear to be missing.** The files contain entry data (the
dictionary entries themselves) but very few annotation labels. Ask:

- "Where are the actual labels — were they recorded in a different file
  format or location?"
- "Were these annotations completed, or was the effort abandoned partway?"
- "What do the numeric columns in the headers represent (e.g., '4',
  '127', '0')?"

### If `n_cells_filled` (formatting) is high

**Labels may be encoded as cell formatting.** Ask:

- "Were colored cells, strikethrough, or other formatting used to indicate
  whether an entry was correct or incorrect?"
- "Is there a key explaining what each color or formatting style means?"

### If `n_kalimat_asal_filled` is meaningful (hundreds per file)

**Some sheets have parcor content embedded in billex sheets.** Ask:

- "Were these files supposed to evaluate billex entries, parcor entries,
  or both?"
- "If both, was there a separate per-row label for billex vs. parcor
  judgment?"

### Universal questions regardless of findings

- "What schema were annotators trained on? Where is the briefing document?"
- "Was IAA computed on these files, and if so, where are the κ scores?"
- "Why are the same dicts (54, 71, 89) annotated in both Tahap 1 and
  Tahap 2, were these re-annotations, or different aspects?"

## 6. Reading guide

Before doing anything with these files, get definitive answers to the
above. Specifically: **if `n_rows_with_named_extras_filled` is < 5% of
`n_rows`, treat the files as effectively unannotated** until proven
otherwise. Building a "cleanup" script on top of the assumption that
labels exist when they don't would be wasted effort.

If after the conversation it turns out:

- **The labels exist in another file/format**: Get those files first;
  these `.xlsx` files might just be the entry-data side and a separate
  label file pairs with them.
- **The annotation effort was abandoned**: Don't use them. Stick with
  the new annotation plan only.
- **The labels are encoded visually**: A different cleanup script (one
  that reads cell formatting) is needed.
- **The labels really are mostly absent and that's the actual state**:
  You can use the entries that DO have labels as a small additional
  validation set, but it won't be a major contribution.

The diagnostic CSV at `_summary.csv` gives you the numbers to take into
that conversation.